# FSA and WHO Nutrition Scoring Examples

This notebook shows how the standalone `nutrition_estimator` package calculates food-item and meal-level healthiness scores.

## Example Item

We use one restaurant-style item with per-serving nutrition fields.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root.name != "nutrition-estimator" and repo_root.parent != repo_root:
    repo_root = repo_root.parent
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from nutrition_estimator import estimate_food, estimate_meal

omelet_bites = {
    "name": "Dunkin Omelet Bites",
    "nutrition": {
        "calories": 180,
        "protein_g": 13,
        "carbs_g": 7,
        "sugar_g": 2,
        "sodium_mg": 460,
        "fat_g": 11,
        "saturated_fat_g": 5,
        "fiber_g": 1,
    },
}

omelet_bites

{'name': 'Dunkin Omelet Bites',
 'nutrition': {'calories': 180,
               'protein_g': 13,
               'carbs_g': 7,
               'sugar_g': 2,
               'sodium_mg': 460,
               'fat_g': 11,
               'saturated_fat_g': 5,
               'fiber_g': 1}}

## FSA Calculation

FSA uses sugar, sodium/salt, fat, and saturated fat. Each component receives a traffic-light score: green = 1, amber = 2, red = 3. The total range is 4-12, and lower is healthier.

In [2]:
estimate_food(omelet_bites, estimation_method="fsa")

{'item': 'Dunkin Omelet Bites',
 'estimated_value': 7,
 'total': 7,
 'status': 'estimated',
 'estimation_method': 'fsa',
 'method': 'FSA traffic-light score from sugar, sodium/salt, fat, and '
           'saturated fat',
 'range': '4-12',
 'direction': 'lower_is_healthier',
 'basis': 'provided',
 'missing_fields': [],
 'components': {'sugar': 1, 'salt': 2, 'fat': 2, 'saturated_fat': 2},
 'normalized_inputs': {'sugar_g': 2.0,
                       'salt_g': 1.15,
                       'fat_g': 11.0,
                       'saturated_fat_g': 5.0}}

## WHO Calculation

WHO-style scoring checks seven nutrient conditions. The range is 0-7, and higher is healthier.

In [3]:
estimate_food(omelet_bites, estimation_method="who")

{'item': 'Dunkin Omelet Bites',
 'estimated_value': 4,
 'total': 4,
 'status': 'estimated',
 'estimation_method': 'who',
 'method': 'WHO-style nutrient-range score from protein, carbohydrates, sugar, '
           'sodium, fat, saturated fat, and fiber',
 'range': '0-7',
 'direction': 'higher_is_healthier',
 'basis': 'provided',
 'missing_fields': [],
 'components': {'protein': 1,
                'carbohydrates': 1,
                'sugar': 1,
                'sodium': 1,
                'fat': 0,
                'saturated_fat': 0,
                'fiber': 0},
 'normalized_inputs': {'calories': 180.0,
                       'protein_g': 13.0,
                       'carbs_g': 7.0,
                       'sugar_g': 2.0,
                       'sodium_mg': 460.0,
                       'fat_g': 11.0,
                       'saturated_fat_g': 5.0,
                       'fiber_g': 1.0,
                       'fat_energy_share': 0.55,
                       'saturated_fat_energy_share': 0.

## Meal-Level Aggregation

Meal-level scoring follows the MealRec+ idea: calculate each item/course score first, then average item scores.

In [4]:
snackin_bacon = {
    "name": "Dunkin Snackin Bacon",
    "nutrition": {
        "calories": 272,
        "protein_g": 7,
        "carbs_g": 10,
        "sugar_g": 10.4,
        "sodium_mg": 378,
        "fat_g": 23,
        "saturated_fat_g": 8,
        "fiber_g": 0.1,
    },
}

iced_coffee = {
    "name": "Dunkin Iced Coffee",
    "nutrition": {
        "calories": 158,
        "protein_g": 1,
        "carbs_g": 29,
        "sugar_g": 33,
        "sodium_mg": 98,
        "fat_g": 3,
        "saturated_fat_g": 2,
        "fiber_g": 0,
    },
}

estimate_meal([omelet_bites, snackin_bacon, iced_coffee], estimation_method="all")

{'estimation_method': 'all',
 'items': [{'item': 'Dunkin Omelet Bites',
            'estimation_method': 'all',
            'status': 'estimated',
            'scores': {'fsa': {'estimated_value': 7,
                               'total': 7,
                               'status': 'estimated',
                               'estimation_method': 'fsa',
                               'method': 'FSA traffic-light score from sugar, '
                                         'sodium/salt, fat, and saturated fat',
                               'range': '4-12',
                               'direction': 'lower_is_healthier',
                               'basis': 'provided',
                               'missing_fields': [],
                               'components': {'sugar': 1,
                                              'salt': 2,
                                              'fat': 2,
                                              'saturated_fat': 2},
                           

## Missing Data / No-Guess Behavior

By default, the estimator does not guess missing nutrition values. If required fields are missing, the score is `None` and the status is `insufficient_data`.


In [5]:
incomplete_item = {
    "nutrition": {
        "sugar_g": 3
    }
}

estimate_food(incomplete_item, estimation_method="fsa")

{'item': None,
 'estimated_value': None,
 'total': None,
 'status': 'insufficient_data',
 'estimation_method': 'fsa',
 'method': 'FSA traffic-light score from sugar, sodium/salt, fat, and '
           'saturated fat',
 'range': '4-12',
 'direction': 'lower_is_healthier',
 'basis': 'provided',
 'missing_fields': ['fat_g', 'saturated_fat_g', 'sodium_mg_or_salt_g'],
 'components': {}}